# 🛰️🔆 TKG Solar — Colab Training (DKASC Alice Springs)

Tái lập **TKGSolarModel** + baselines trên dữ liệu co-located **DKASC Alice Springs (2020–2022, 5-min) + Himawari-8**.

> **Runtime → Change runtime type → GPU.** Chạy tuần tự từ trên xuống.

**Hai giai đoạn:**
- 🟦 **Phase A — Baselines** (Persistence, ARIMA, LSTM, GRU, Transformer, Temporal-GNN): chỉ cần **`alice_2020_2022_clean.csv`**, KHÔNG cần Himawari → chạy ngon trên **Colab FREE (T4)**.
- 🟧 **Phase B — Proposed (TKG)**: cần thêm **`himawari_alice/frames.h5`** (full 2020–2022) → cần GPU mạnh (A100); ViT-224 rất nặng trên T4.

Upload lên Drive (vì `data/` không có trên GitHub):
```
<DATA_ROOT>/dkasc/alice_2020_2022_clean.csv   # Phase A + B (18 MB)  ← CHỈ CẦN FILE NÀY cho baselines
<DATA_ROOT>/himawari_alice/frames.h5          # chỉ Phase B (~1.2 GB)
```
> `synthetic_array_*.csv` KHÔNG cần (chỉ là fixture test).

<hr style="border:none;border-top:3px solid #d95f0e">
## 1 · GPU

In [ ]:
!nvidia-smi

<hr style="border:none;border-top:3px solid #d95f0e">
## 2 · Clone repo (nhánh `init-project`)

In [ ]:
import os
if not os.path.isdir('tkg-solar-satellite-reasoning'):
    !git clone https://github.com/DucLong06/tkg-solar-satellite-reasoning.git
%cd tkg-solar-satellite-reasoning
!git checkout init-project && git pull

<hr style="border:none;border-top:3px solid #d95f0e">
## 3 · Dependencies (statsmodels cho ARIMA)

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # set TRƯỚC khi torch chạm CUDA
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available())
!pip install -q timm torch_geometric h5py scikit-learn pyyaml tqdm statsmodels

<hr style="border:none;border-top:3px solid #d95f0e">
## 4 · Dữ liệu (mount Drive)
Điền `DATA_ROOT`. **Phase A chỉ cần file DKASC CSV** — Himawari có thể thiếu (Phase B mới cần).

In [ ]:
import pathlib
from google.colab import drive
drive.mount('/content/drive')

# <-- ĐIỀN đường dẫn thư mục dữ liệu trên Drive:
DATA_ROOT = '/content/drive/MyDrive/tkg-solar-data'

DKASC_CSV    = f'{DATA_ROOT}/dkasc/alice_2020_2022_clean.csv'   # bắt buộc (Phase A + B)
HIMAWARI_DIR = f'{DATA_ROOT}/himawari_alice'                    # chỉ Phase B
print(('OK  ' if pathlib.Path(DKASC_CSV).exists() else 'MISSING '), DKASC_CSV)
print(('OK  ' if pathlib.Path(f'{HIMAWARI_DIR}/frames.h5').exists() else '(chưa có, Phase B mới cần) '), HIMAWARI_DIR)

<hr style="border:none;border-top:3px solid #d95f0e">
## 5 · Config + thiết bị
`paper_config.yaml` đã trỏ DKASC + split cố định (2020-01→2021-09 / 2021-10→12 / 2022). `resolve_runtime` tự chọn precision theo GPU.

> 💡 **T4 free** không hỗ trợ bf16 → tự dùng **fp16**. ViT-224 (Phase B) rất nặng trên T4 → nên để Phase B cho A100; T4 tập trung **Phase A**.

In [ ]:
from src.training.config import Config
from src.training.gpu_autoscale import resolve_runtime
from src.common.seeding import seed_everything
from src.common.shapes import EMBED_DIM, FUSION_DIM, HORIZON_MINUTES, N_HORIZONS, N_METEO_FEATURES

cfg = Config.from_yaml('configs/paper_config.yaml')
cfg.device = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg.dkasc_csv = DKASC_CSV
cfg.himawari_dir = HIMAWARI_DIR
cfg.checkpoint_dir = f'{DATA_ROOT}/checkpoints'
cfg.cache_dir = 'data/cache'
resolve_runtime(cfg)              # T4: fp16; A100: bf16 + auto batch; cpu: no-op
seed_everything(cfg.seed)

print('device   :', cfg.device, '| precision', cfg.precision, '| batch', cfg.batch_size)
print('contract :', N_METEO_FEATURES, 'meteo feats | horizons', HORIZON_MINUTES, 'min | PV kW')
print('split    :', cfg.train_end, '/', cfg.val_end, '| min_steps', cfg.min_steps)

<hr style="border:none;border-top:3px solid #d95f0e">
## 6 · Helper resume/skip checkpoint
`last_{tên}.pt` trên Drive: chưa có→train mới; dang dở→resume; xong→hỏi skip/retrain.

In [ ]:
from pathlib import Path

def ckpt_status(label):
    p = Path(cfg.checkpoint_dir) / f'last_{label}.pt'
    if not p.exists():
        return 'none'
    try:
        ck = torch.load(p, map_location='cpu', weights_only=False)
    except Exception:
        return 'none'
    return 'done' if ck.get('done') else 'partial'

def decide(label):
    st = ckpt_status(label)
    if st == 'none':
        return 'fresh'
    if st == 'partial':
        print(f'[{label}] checkpoint dang dở -> resume'); return 'resume'
    ans = input(f"[{label}] đã train XONG. 'r'=train lại, Enter=bỏ qua: ").strip().lower()
    return 'fresh' if ans == 'r' else 'skip'

print('decide(tên) -> fresh | resume | skip')

<hr style="border:none;border-top:4px solid #2c7fb8">

# 🟦 PHASE A — Baselines (KHÔNG cần Himawari)

`use_satellite=False` → pipeline canh PV+meteo trên **full DKASC 2020–2022**. Frame vệ tinh = 0 (baselines bỏ qua) nên dùng `img_size=8` cho nhẹ. Chạy ngon trên **T4 free**.

## A1 · Pipeline (PV + meteo, full DKASC)

In [ ]:
from src.data_pipeline import DataPipeline

splits_base = DataPipeline.load(
    cfg.dkasc_csv, cfg.himawari_dir,
    k=cfg.k, batch_size=256, img_size=8,        # sat=0 -> img_size nhỏ; batch lớn cho baselines nhẹ
    min_steps=cfg.min_steps,
    train_end=cfg.train_end, val_end=cfg.val_end,
    cadence_min=cfg.cadence_min, night_ghi_thresh=cfg.night_ghi_thresh,
    cache_dir=cfg.cache_dir, use_satellite=False, scaler_out=cfg.scaler_out,
)
for k_, v_ in splits_base.meta.items():
    print(f'  {k_:18s}: {v_}')

## A2 · Train 6 baselines (cùng split/seed)
Persistence/ARIMA = eval-only. ⚠️ **ARIMA rất chậm** trên test lớn — bỏ khỏi `BASELINES` nếu muốn nhanh.

In [ ]:
import copy
from src.evaluation.baseline_models import build_baseline, is_eval_only
from src.lstm_baseline.lstm_forecaster import LSTMForecaster
from src.training.train_loop import fit
from src.training.evaluate import evaluate_model

bcfg = copy.copy(cfg); bcfg.batch_size = 256
# bcfg.epochs = 50   # giảm để thử nhanh
BASELINES = ['Persistence', 'ARIMA', 'LSTM', 'GRU', 'Transformer', 'Temporal-GNN']
# BASELINES = [m for m in BASELINES if m != 'ARIMA']   # bỏ comment -> nhanh hơn nhiều
base_results = {}

def build_base(name):
    if name == 'LSTM':
        return LSTMForecaster(hidden_dim=bcfg.embed_dim, dropout=bcfg.dropout)
    return build_baseline(name, bcfg)

def run_base(name):
    import gc; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    seed_everything(bcfg.seed)
    model_b = build_base(name)
    if is_eval_only(model_b):
        print(f'== {name} (eval-only) =='); model_b.to(bcfg.device)
    else:
        action = decide(name)
        if action == 'skip':
            best = Path(bcfg.checkpoint_dir) / f'best_{name}.pt'
            model_b.load_state_dict(torch.load(best, map_location=bcfg.device, weights_only=True)['model_state'])
            model_b.to(bcfg.device)
        else:
            if action == 'fresh':
                (Path(bcfg.checkpoint_dir) / f'last_{name}.pt').unlink(missing_ok=True)
            fit(model_b, splits_base, bcfg, verbose=True, desc=name, resume=True)
    m = evaluate_model(model_b, splits_base.test_loader, splits_base.scalers, bcfg.device, bcfg.mape_min_value)
    base_results[name] = m['overall']
    print(f"{name}: MAE={m['overall']['mae']:.4f} RMSE={m['overall']['rmse']:.4f} MAPE={m['overall']['mape']:.2f}%")
    model_b.to('cpu'); del model_b

for _n in BASELINES:
    run_base(_n)

## A3 · Bảng xếp hạng baselines (kW)

In [ ]:
from src.evaluation.benchmark_table import build_benchmark, relative_ordering
print(build_benchmark(base_results, scalers=splits_base.scalers))
print('\nThứ tự (tốt→kém theo MAE):', ' < '.join(relative_ordering(base_results)))

<hr style="border:none;border-top:4px solid #d95f0e">

# 🟧 PHASE B — Proposed (TKG) — cần Himawari

Chỉ chạy khi đã upload `himawari_alice/frames.h5` (full 2020–2022). ViT-224 nặng → **nên dùng A100**; T4 thì bật `cfg.freeze_backbone=True` + chấp nhận chậm.

> ⚠️ Himawari **một phần** (vài tháng) → split cố định rỗng. Dùng split phân số: thay `train_end/val_end` bằng `train_frac=.7, val_frac=.15` và hạ `min_steps`.

## B1 · Pipeline (PV + meteo + vệ tinh)

In [ ]:
assert pathlib.Path(f'{HIMAWARI_DIR}/frames.h5').exists(), 'Thiếu frames.h5 -> upload Himawari trước'
# cfg.freeze_backbone = True   # bỏ comment trên T4 để nhẹ ViT
splits_full = DataPipeline.load(
    cfg.dkasc_csv, cfg.himawari_dir,
    k=cfg.k, batch_size=cfg.batch_size, img_size=cfg.img_size,
    min_steps=cfg.min_steps,
    train_end=cfg.train_end, val_end=cfg.val_end,
    cadence_min=cfg.cadence_min, night_ghi_thresh=cfg.night_ghi_thresh,
    cache_dir=cfg.cache_dir, use_satellite=True, scaler_out=cfg.scaler_out,
)
print(splits_full.meta)

## B2 · Train Proposed (TKGSolarModel)

In [ ]:
from src.fusion_predictor.tkg_solar_model import TKGSolarModel

model = TKGSolarModel.from_config(cfg)
print('FUSION_DIM', FUSION_DIM, '= 3 x', EMBED_DIM,
      '| params', f'{sum(p.numel() for p in model.parameters()):,}')

action = decide('full')
if action == 'skip':
    ck = torch.load(Path(cfg.checkpoint_dir) / 'last_full.pt', map_location=cfg.device, weights_only=False)
    model.load_state_dict(ck['model_state']); model.to(cfg.device)
    history = ck['history']; history['best_val_mae'] = ck['best_val']
    history['best_checkpoint'] = str(Path(cfg.checkpoint_dir) / 'best_full.pt')
else:
    if action == 'fresh':
        (Path(cfg.checkpoint_dir) / 'last_full.pt').unlink(missing_ok=True)
    history = fit(model, splits_full, cfg, verbose=True, resume=True)
print('\nbest val-MAE:', round(history['best_val_mae'], 5))

## B3 · Đánh giá Proposed (theo horizon, kW)

In [ ]:
from src.training.train_loop import predict_loader
from src.metrics.regression_metrics import compute_per_horizon

yt, yp = predict_loader(model, splits_full.test_loader, cfg.device)
yt = splits_full.scalers.inverse_pv(yt.numpy()); yp = splits_full.scalers.inverse_pv(yp.numpy())
per = compute_per_horizon(yt, yp, HORIZON_MINUTES, cfg.mape_min_value)
print(f"{'horizon':>10} | {'MAE':>9} | {'RMSE':>9} | {'MAPE %':>9}")
print('-'*46)
for lab in ['overall', *[f'{m}min' for m in HORIZON_MINUTES]]:
    m = per[lab]
    print(f"{lab:>10} | {m['mae']:9.4f} | {m['rmse']:9.4f} | {m['mape']:9.2f}")

## B4 · Bảng tổng + ablation (tùy chọn)

In [ ]:
from src.evaluation.benchmark_table import build_ablation_table

all_results = dict(base_results); all_results['Proposed'] = per['overall']
print(build_benchmark(all_results, scalers=splits_full.scalers))
print('\nThứ tự:', ' < '.join(relative_ordering(all_results)))

RUN_ABLATION = False   # True -> chạy ablation (tốn thêm ~3-4 lần train Proposed)
if RUN_ABLATION:
    abl = {}
    for label, ov in [('Full', {}), ('-sat', {'use_sat': False}),
                      ('-graph', {'use_graph': False}), ('-meteo', {'use_meteo': False})]:
        acfg = copy.copy(cfg)
        acfg.use_sat = acfg.use_meteo = acfg.use_graph = True
        for kk, vv in ov.items(): setattr(acfg, kk, vv)
        seed_everything(acfg.seed)
        am = TKGSolarModel.from_config(acfg)
        fit(am, splits_full, acfg, verbose=True, desc=f'abl-{label}', resume=False)
        abl[label] = evaluate_model(am, splits_full.test_loader, splits_full.scalers, acfg.device, acfg.mape_min_value)['overall']
        am.to('cpu'); del am
    print(build_ablation_table(abl))

<hr style="border:none;border-top:3px solid #d95f0e">
## 7 · Lưu kết quả về Drive

In [ ]:
import shutil
SAVE_DIR = f'{DATA_ROOT}/outputs'
pathlib.Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
if 'history' in globals():
    shutil.copy(history['best_checkpoint'], SAVE_DIR)
if pathlib.Path(cfg.scaler_out).exists():
    shutil.copy(cfg.scaler_out, SAVE_DIR)
cfg.save(f'{SAVE_DIR}/resolved_config.yaml')
print('Đã lưu ->', SAVE_DIR)